# ATCoder — Graph and Spectral Analysis

Graph/spectral coverage plus sampled PSS and Wasserstein score distributions.


In [ ]:
# SPECTRA notebook progress: begin
from tqdm.auto import tqdm

def _spectra_enable_cell_progress():
    from IPython import get_ipython
    _shell = get_ipython()
    if _shell is None:
        return
    _old = getattr(_shell, "_spectra_cell_progress", None)
    if _old is not None:
        _old[0].close()
        for _event, _callback in _old[1:]:
            _shell.events.unregister(_event, _callback)
    _bar = tqdm(total=7, desc="Notebook cells", unit="cell", mininterval=2)
    def _after_cell(_result):
        if _bar.n < _bar.total:
            _bar.update(1)
        if getattr(_result, "error_in_exec", None) or getattr(_result, "error_before_exec", None):
            _bar.set_description("Notebook cell failed")
            _bar.close()
        elif _bar.n >= _bar.total:
            _bar.close()
    _shell.events.register("post_run_cell", _after_cell)
    _shell._spectra_cell_progress = (_bar, ("post_run_cell", _after_cell))

_spectra_enable_cell_progress()
# SPECTRA notebook progress: end

from pathlib import Path
import sys
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'spectral_code').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from the repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import importlib
import spectral_code.evaluation.atcoder_analysis as atcoder_analysis
importlib.reload(atcoder_analysis)
from spectral_code.evaluation.atcoder_analysis import *

sns.set_theme(style='whitegrid', context='notebook')
ROOT = clean_data_dir()
ROOT


In [ ]:
coverage, summary = graph_coverage(ROOT)
summary
coverage


In [ ]:
sns.catplot(data=coverage, x='graph_type', y='codes', hue='spectral_status', col='language', kind='bar'); plt.show()


## Clone and non-clone examples: code, graph, and spectrum


In [ ]:
examples = select_pair_examples(ROOT, graph_type='cpg', per_label=2, seed=42)
for example in examples:
    display(pd.DataFrame([example['pair']]))
    display(plot_pair_code_graph_spectra(ROOT, example, graph_type='cpg', max_code_lines=32, max_graph_nodes=80))


## Balanced train-pair sample


In [ ]:
scores = sample_similarity_scores(ROOT, sample_size=10_000, seed=42)
scores.groupby(['graph_type', 'label_name']).size().rename('scores').to_frame()


In [ ]:
sns.displot(data=scores, x='pss', hue='label_name', col='graph_type', kind='hist', stat='density', common_norm=False, bins=40, col_wrap=2); plt.show()


In [ ]:
sns.displot(data=scores, x='wasserstein', hue='label_name', col='graph_type', kind='hist', stat='density', common_norm=False, bins=40, col_wrap=2); plt.show()
